# NaijaBuddy — Dense Dataset Preparation

Runs the data acquisition on **Colab's bandwidth**, not yours.

**What it does**
1. Streams large samples of Yelp, Goodreads and Amazon (Books) reviews via DuckDB.
2. Extracts a genuine dense **bipartite k-core** from each — every user *and* every item has at least `k` interactions (default k=3). This is what makes NDCG / HitRate meaningful.
3. Caps each domain to its ~350 densest users, then re-cores so the result is still a valid k-core.
4. Exports three small CSVs and zips them for download.

**All three sources are HuggingFace parquet datasets**, so `LIMIT` reads only the head of shard 0 — no slow full-file scans.

**How to use**
1. `Runtime -> Run all`.
2. When the last cell finishes it downloads `naijabuddy_dense_data.zip`.
3. Unzip it so the repo has `data/yelp_dense.csv`, `data/goodreads_dense.csv`, `data/amazon_dense.csv`.
4. Locally run `python data_enricher.py` — it reads the CSVs, **no network needed**.

In [ ]:
!pip install -q duckdb

In [ ]:
import duckdb, time, os, zipfile
import requests
import pandas as pd

STREAM_RETRIES = 5
STREAM_RETRY_WAIT_S = 4

def hf_parquet_urls(dataset):
    # Look up the auto-converted parquet shard URLs for a HuggingFace dataset
    # via the dataset-viewer API. Robust to shard count / numbering changes.
    r = requests.get(
        f"https://datasets-server.huggingface.co/parquet?dataset={dataset}",
        timeout=60,
    )
    r.raise_for_status()
    return [f["url"] for f in r.json()["parquet_files"]]

def stream_query(query, label):
    # Retry the whole query. A corrupted compressed chunk surfaces as a
    # Snappy/LZ4 decompression error AFTER the HTTP read completes, which
    # DuckDB's http_retries setting cannot catch.
    last_err = None
    for attempt in range(1, STREAM_RETRIES + 1):
        con = duckdb.connect()
        try:
            con.execute("SET http_retries=3;")
            con.execute("SET http_timeout=120000;")
            df = con.execute(query).fetchdf()
            con.close()
            print(f"  [{label}] streamed {len(df)} rows")
            return df
        except Exception as e:
            last_err = e
            try:
                con.close()
            except Exception:
                pass
            print(f"  [{label}] attempt {attempt}/{STREAM_RETRIES} failed: {str(e).splitlines()[0]}")
            if attempt < STREAM_RETRIES:
                time.sleep(STREAM_RETRY_WAIT_S)
    raise last_err

def true_kcore(df, k):
    # Iterative k-core to a fixed point: repeatedly drop users and items with
    # fewer than k interactions until the set stops changing.
    while len(df):
        uc = df["user_id"].value_counts()
        ic = df["item_id"].value_counts()
        keep = df[df["user_id"].map(uc).ge(k) & df["item_id"].map(ic).ge(k)]
        if len(keep) == len(df):
            break
        df = keep
    return df

def densify(df, k=3, limit_users=2000, label=""):
    if df.empty:
        print(f"  [{label}] empty input")
        return df
    # 1. strongest k-core the sample supports (relax k if it wipes everything out)
    core, eff_k = df.iloc[0:0], 0
    for kk in range(k, 0, -1):
        c = true_kcore(df, kk)
        if len(c):
            core, eff_k = c, kk
            break
    # 2. cap to the densest limit_users users, then re-core so it stays a k-core
    if limit_users and len(core):
        top = core["user_id"].value_counts().nlargest(limit_users).index
        core = core[core["user_id"].isin(top)]
        for kk in range(eff_k, 0, -1):
            c = true_kcore(core, kk)
            if len(c):
                core, eff_k = c, kk
                break
    print(f"  [{label}] final {eff_k}-core: {len(core)} reviews, "
          f"{core['user_id'].nunique()} users, {core['item_id'].nunique()} items")
    return core


In [ ]:
# --- Yelp: HuggingFace parquet, head of shard 0 (no scan) ---
YELP_SQL = """
WITH s AS (
  SELECT user_id, business_id, stars, text
  FROM read_parquet('https://huggingface.co/api/datasets/yashraizada/yelp-open-dataset-reviews/parquet/default/train/0.parquet')
  WHERE stars > 0 LIMIT 250000
)
SELECT s.user_id AS user_id, s.business_id AS item_id, b.name AS item_name,
       b.categories AS category, s.stars AS rating, s.text AS review_text
FROM s JOIN read_parquet('https://huggingface.co/api/datasets/yashraizada/yelp-open-dataset-business/parquet/default/train/0.parquet') b
  ON s.business_id = b.business_id
"""

# --- Goodreads: HuggingFace parquet, head of shard 0 (no scan) ---
GOODREADS_SQL = """
WITH s AS (
  SELECT user_id, book_id, rating, review_text
  FROM read_parquet('https://huggingface.co/api/datasets/vngclinh/goodreads-reviews/parquet/default/train/0.parquet')
  WHERE rating > 0 LIMIT 250000
)
SELECT s.user_id AS user_id, s.book_id AS item_id,
       COALESCE(m.Book, 'Book #' || s.book_id) AS item_name,
       'Goodreads (Book)' AS category, s.rating AS rating, s.review_text AS review_text
FROM s LEFT JOIN read_parquet('https://huggingface.co/api/datasets/Eitanli/goodreads/parquet/default/train/0.parquet') m
  ON s.book_id = regexp_extract(m.URL, '/book/show/([0-9]+)', 1)
"""

# --- Amazon (Books): HuggingFace parquet mirror of McAuley 2023 ---
# The original ClickHouse S3 file needed a multi-GB "WHERE category='Books'"
# scan -- roughly 50x slower from Colab. This dataset is Books-only, so LIMIT
# just reads the head of shard 0. Book titles come from the metadata
# companion; selecting only parent_asin+title keeps that read small (parquet
# column projection skips the heavy description/images/videos columns).
amz_review_urls = hf_parquet_urls("cogsci13/Amazon-Reviews-2023-Books-Review")
amz_meta_urls = hf_parquet_urls("cogsci13/Amazon-Reviews-2023-Books-Meta")
print(f"Amazon: {len(amz_review_urls)} review shards, {len(amz_meta_urls)} meta shards discovered")
amz_meta_list = "[" + ",".join(f"'{u}'" for u in amz_meta_urls) + "]"
AMAZON_SQL = f"""
WITH s AS (
  SELECT user_id, parent_asin, rating, text
  FROM read_parquet('{amz_review_urls[0]}')
  WHERE rating > 0 AND user_id IS NOT NULL AND parent_asin IS NOT NULL
  LIMIT 600000
)
SELECT s.user_id AS user_id,
       s.parent_asin AS item_id,
       COALESCE(m.title, 'Amazon Book ' || s.parent_asin) AS item_name,
       'Amazon (Book)' AS category,
       s.rating AS rating,
       s.text AS review_text
FROM s
LEFT JOIN (SELECT parent_asin, title FROM read_parquet({amz_meta_list})) m
  ON s.parent_asin = m.parent_asin
"""

print("=== Yelp ===")
yelp = densify(stream_query(YELP_SQL, "Yelp"), label="Yelp")
print("=== Goodreads ===")
goodreads = densify(stream_query(GOODREADS_SQL, "Goodreads"), label="Goodreads")
print("=== Amazon (Books) ===")
amazon = densify(stream_query(AMAZON_SQL, "Amazon"), label="Amazon")

In [ ]:
COLS = ["user_id", "item_id", "item_name", "category", "rating", "review_text"]
os.makedirs("data", exist_ok=True)

for name, df in [("yelp", yelp), ("goodreads", goodreads), ("amazon", amazon)]:
    out = df[COLS].copy()
    out["review_text"] = out["review_text"].fillna("")
    out["item_name"] = out["item_name"].fillna("")
    out.to_csv(f"data/{name}_dense.csv", index=False)
    kb = os.path.getsize(f"data/{name}_dense.csv") / 1024
    print(f"data/{name}_dense.csv  ->  {len(out)} rows, {kb:.0f} KB")

with zipfile.ZipFile("naijabuddy_dense_data.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["yelp", "goodreads", "amazon"]:
        z.write(f"data/{name}_dense.csv")

print("Done. Downloading naijabuddy_dense_data.zip ...")
from google.colab import files
files.download("naijabuddy_dense_data.zip")